### students performance Classification:
This notebook loads data and prepares the data.

In [1]:
# ================================
# 1. MOUNT GOOGLE DRIVE
# ================================
from google.colab import drive
# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Read the saved data
print("=" * 70)
print("READING SAVED DATA")
print("=" * 70)

try:
    df = pd.read_csv(r"/content/drive/MyDrive/Colab Notebooks/teachers and students SRQ_total and students performance analysis/processed_dataset2.xls")
    print(f"✅ Dataset loaded successfully!")
    print(f" Shape: {df.shape}")
    print(f" Columns: {len(df.columns)}")
    print(f" Total records: {len(df)}")
except FileNotFoundError:
    print(" File 'processed_data.csv' not found.")
    exit()

READING SAVED DATA
✅ Dataset loaded successfully!
 Shape: (758, 64)
 Columns: 64
 Total records: 758


In [3]:
# Filter students
students_df = df[df["service_year_teacher"].isnull()].copy()

# Drop irrelevant columns
drop_cols = ["service_year_teacher", "OSLO1", "OSLO2", "OSLO3"]
students_df = students_df.drop(columns=[c for c in drop_cols if c in students_df.columns])

# Identify high-null columns
nulls = students_df.isnull().sum()
high_null_auto = nulls[nulls > len(students_df)*0.50].index.tolist()

# Predefined high-null cols
high_null_candidates = [
    'Alcohol_2','Alcohol_3','Alcohol_4','Alcohol_5','Alcohol_6','alcohol_7',
    'tobaco_2','tobaco_3','tobaco_4','tobaco_5','tobaco_6','tobaco_7',
    'khat_2','khat_3','khat_4','khat_5','khat_6','khat_7'
]

final_high_null = list(set(high_null_candidates + high_null_auto))
students_df = students_df.drop(columns=[c for c in final_high_null if c in students_df.columns])

# Fill academic + substance columns (numeric)
numeric_fill = ['average', 'rank']
binary_cat_fill = ['Alcohol_1', 'tobaco_1', 'khat_1']

for col in numeric_fill:
    if col in students_df.columns:
        median_value = students_df[col].median()
        students_df[col] = students_df[col].fillna(median_value)

for col in binary_cat_fill:
    if col in students_df.columns:
        mode_value = students_df[col].mode()[0]
        students_df[col] = students_df[col].fillna(mode_value)

# Fill remaining columns based on type
for col in students_df.columns:
    if students_df[col].isnull().sum() > 0:
        if np.issubdtype(students_df[col].dtype, np.number):
            students_df[col] = students_df[col].fillna(students_df[col].median())
        else:
            students_df[col] = students_df[col].fillna(students_df[col].mode()[0])

# Create SRQ_total
srq_cols = [f"SRQ{i}" for i in range(1, 21)]
srq_used = [c for c in srq_cols if c in students_df.columns]

if srq_used:
    students_df["SRQ_total"] = students_df[srq_used].sum(axis=1)
    students_df = students_df.drop(columns=srq_used)

# Create MPSS_total
mpss_cols = [f"MPSS{i}" for i in range(1, 13)]
mpss_used = [c for c in mpss_cols if c in students_df.columns]

if mpss_used:
    students_df["MPSS_total"] = students_df[mpss_used].sum(axis=1)
    students_df = students_df.drop(columns=mpss_used)

print("Cleaned student dataset shape:", students_df.shape)


Cleaned student dataset shape: (620, 12)


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from scipy.stats import mstats
import warnings
warnings.filterwarnings('ignore')

#----------------------------------------------------------
# ENCODING
#----------------------------------------------------------
students_encoded = students_df.copy()
label_encoders = {}

# One-hot encode school
school_dummies = pd.get_dummies(students_encoded['school'], prefix='school')
students_encoded = pd.concat([students_encoded.drop('school', axis=1), school_dummies], axis=1)

# Label encode binary/ordinal
categorical = ['sex', 'Education', 'Alcohol_1', 'tobaco_1', 'khat_1']
for col in categorical:
    if col in students_encoded.columns:
        le = LabelEncoder()
        students_encoded[col] = le.fit_transform(students_encoded[col].astype(str))
        label_encoders[col] = le

#----------------------------------------------------------
# WINSORIZATION (Outliers)
#----------------------------------------------------------
num_features = ['age', 'SRQ_total', 'MPSS_total', 'average']
num_features = [c for c in num_features if c in students_encoded.columns]

for feature in num_features:
    data = students_encoded[feature]
    students_encoded[feature] = mstats.winsorize(data, limits=[0.05, 0.05])

#----------------------------------------------------------
# TARGETS
#----------------------------------------------------------
rank_33 = students_encoded['rank'].quantile(0.33)
rank_66 = students_encoded['rank'].quantile(0.66)

students_encoded['rank_three_class'] = pd.cut(
    students_encoded['rank'],
    bins=[-np.inf, rank_33, rank_66, np.inf],
    labels=[0, 1, 2]
)

#----------------------------------------------------------
# FEATURE SELECTION FUNCTION
#----------------------------------------------------------
def select_features(X, y, mode='reg', k=6):
    scores = {}

    if mode == 'reg':
        fsel = SelectKBest(f_regression, k='all').fit(X, y)
        misel = SelectKBest(mutual_info_regression, k='all').fit(X, y)
        rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y)
    else:
        fsel = SelectKBest(f_classif, k='all').fit(X, y)
        misel = SelectKBest(mutual_info_classif, k='all').fit(X, y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)

    scores['f'] = pd.Series(fsel.scores_, index=X.columns).sort_values(ascending=False)
    scores['mi'] = pd.Series(misel.scores_, index=X.columns).sort_values(ascending=False)
    scores['rf'] = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

    corr = X.apply(lambda col: abs(col.corr(y)))
    scores['corr'] = corr.sort_values(ascending=False)

    selected = set()
    for key in scores:
        top = scores[key].dropna().head(k).index.tolist()
        selected.update(top)

    return list(selected)

#----------------------------------------------------------
# APPLY FEATURE SELECTION
#----------------------------------------------------------
base_features = [c for c in students_encoded.columns if c not in ['ID', 'rank', 'rank_three_class']]

# Regression
X_reg = students_encoded[base_features]
y_reg = students_encoded['rank']
reg_features = select_features(X_reg, y_reg, mode='reg', k=6)

# Classification (3-class)
X_clf = students_encoded[base_features]
y_clf = students_encoded['rank_three_class']
clf_features = select_features(X_clf, y_clf, mode='clf', k=6)

#----------------------------------------------------------
# FINAL DATASETS
#----------------------------------------------------------
regression_dataset = students_encoded[reg_features + ['rank']]
three_class_dataset = students_encoded[clf_features + ['rank_three_class']]
